In [ ]:
import os
import s3fs

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = (SparkSession.builder
         .appName('Spark_app')
         .getOrCreate())

In [ ]:
# spark.stop()

In [ ]:
spark = SparkSession.builder \
    .appName("StreamingJob") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [ ]:
spark

In [ ]:
BUCKET = "fgao-ensae"
FILE_KEY_S3 = "Data_spark/Streaming"
s3_path = f"s3a://{BUCKET}/{FILE_KEY_S3}"
s3_path

In [ ]:
# dbutils.fs.ls("FileStore/tables")

# dbutils.fs.rm("FileStore/tables/",True)

In [ ]:
static_df = spark.read.csv(
    s3_path,
    header=True,
    inferSchema= True
)

In [ ]:
static_df.show()
static_df.count()
static_df.printSchema()

## 1. Create a streaming df from file source 

In [ ]:
stream_df = spark.readStream.csv(s3_path, 
                            header = True,
                            schema = static_df.schema)

In [ ]:
stream_df

# attention : we can not add a action to steam dataframe

## 2. Create a normal query 

In [ ]:
from pyspark.sql.functions import year, month, sum

In [ ]:
stock_query = (
    stream_df
    .groupby('Name', year('Date').alias('year'))
    .agg(sum('Volume').alias('Volume_Total'))
)

#  .orderBy("Name", "year", "month") # Attention, Sorting is not supported on streaming DataFrames/Dataset

## 3. Write this query in streaming mode in memory

In [ ]:
# n'oublie pas de donner un nom pour pourvoir la stopper facilement

stream_stock_query = (stock_query.writeStream
                .trigger(processingTime= '10 seconds')
                .format('memory') # Sinks 
                .outputMode('update') # or "complete", "Append" output mode not supported when there are streaming aggregations on streaming DataFrames/DataSets without watermark. See more explication.
                .queryName('stock_results') # nom de TABLE DE RESULTAT DE CETTE REQ streaming
                .start()
    
)


#### trigger
* If no trigger setting is explicitly specified, then by default, the query will be executed in micro-batch mode, where micro-batches will be generated as soon as the previous micro-batch has completed processing.
* .trigger(processingTime='2 seconds') : The query will be executed with micro-batches mode, where micro-batches will be kicked off at the user-specified intervals.

#### Output mode
Other aggregations = > only Complete, Update
Since no watermark is defined (only defined in other category), old aggregation state is not dropped. Append mode is not supported as aggregates can update thus violating the semantics of this mode.

waterwark : https://docs.databricks.com/en/structured-streaming/watermarks.html

Aggregation on event-time with watermark => Complete, Update and append

Append mode uses watermark to drop old aggregation state. This means that as new rows are brought into the table, Spark will only keep around rows that are below the “watermark”. Update mode also uses the watermark to remove old aggregation state. By definition, complete mode does not drop old aggregation state since this mode preserves all data in the Result Table.

#### Memory sink 

The memory sink is a simple source for testing your streaming system. It’s similar to the console sink except that rather than printing to the console, it collects the data to the driver and then makes the data available as an **in-memory table** that is available for interactive querying. This sink is not fault tolerant, and you shouldn’t use it in production, but is great for testing and querying your stream during development. This supports both append and complete output modes: 

// in Scala activityCounts.writeStream.format("memory").queryName("my_device_table") 

If you do want to output data to a table for interactive SQL queries in production, the authors recommend using the Parquet file sink on a distributed file system (e.g., S3). You can then query the data from any Spark application.

In [ ]:
os.listdir()

## 4. Visualize streaming query results in real time

In [ ]:
stream_stock_query.isActive
# stream_query.stop()

In [ ]:
import time

max_duration = 50

start_time = time.time()
# print(start_time)

while stream_query.isActive:
    spark.sql("select * from res_stock order by Name, year").show(100)
    time.sleep(10)

    if time.time() - start_time > max_duration : 
        print("Max duration reached. Stop SQL query.")
        break  # Sortir explicitement de la boucle

if stream_query.isActive : 
    stream_query.stop()

In [ ]:
# ! pip install pandas

In [ ]:
# ! pip install matplotlib

In [ ]:
# Exemple mais le graphique ne se rafraichir pas dans JUPYTER

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np

# Configuration de la durée maximale
max_duration = 100
start_time = time.time()

# Initialisation de la figure Matplotlib
plt.ion()  # Mode interactif pour mise à jour en temps réel
fig, ax = plt.subplots()
ax.set_title("Volume par année et par nom")
ax.set_xlabel("Année")
ax.set_ylabel("Volume total")

# Boucle de streaming
while stream_stock_query.isActive:
    # Exécuter la requête Spark pour récupérer les résultats
    result_df = spark.sql("""
        SELECT Name, year, SUM(Volume_Total) AS Total_Volume
        FROM stock_results
        GROUP BY Name, year
        ORDER BY year, Name
    """)
    pandas_df = result_df.toPandas()  # Convertir en DataFrame Pandas pour visualisation

    # Vérification si les données sont disponibles
    if not pandas_df.empty:
        ax.clear()  # Effacer l'ancien graphique

        # Création d'un graphique groupé
        years = pandas_df["year"].unique()
        names = pandas_df["Name"].unique()
        bar_width = 0.2  # Largeur des barres
        x_positions = np.arange(len(years))  # Positions de base pour les années

        # Tracer les barres pour chaque `Name`
        for i, name in enumerate(names):
            group_data = pandas_df[pandas_df["Name"] == name]
            volumes = [group_data[group_data["year"] == year]["Total_Volume"].sum() for year in years]
            ax.bar(
                x_positions + i * bar_width,  # Décalage horizontal pour chaque groupe
                volumes,
                width=bar_width,
                label=name  # Ajouter une légende pour chaque `Name`
            )

        ax.set_title("Volume par année et par nom")
        ax.set_xlabel("Année")
        ax.set_ylabel("Volume total")
        ax.set_xticks(x_positions + (len(names) - 1) * bar_width / 2)  # Ajuster les positions des ticks
        ax.set_xticklabels(years)  # Étiquettes pour les années
        ax.legend(title="Name")  # Ajouter une légende

        plt.draw()  # Redessiner le graphique
        plt.pause(0.1)  # Pause pour rafraîchir l'affichage

    # Attendre avant la prochaine itération
    time.sleep(10)

    # Arrêter si la durée maximale est atteinte
    if time.time() - start_time > max_duration:
        print("max duration atteint. Stopper la boucle.")
        break

# Arrêter le streaming proprement
if stream_stock_query.isActive:
    stream_stock_query.stop()

# Désactiver le mode interactif et afficher le graphique final
plt.ioff()
plt.show()

In [ ]:
pandas_df